# SocialChoice-05 : Gibbard-Satterthwaite sans mystere - la manipulation comme temoin

> **Twin C# .NET de `GameTheory/SocialChoice/ChoixSocial-05-Gibbard-Satterthwaite-Python.ipynb`** (Python). Kernel `.net-csharp`. Meme fil pedagogique, regles de vote from-scratch en C# (BCL .NET 9 : `System.Linq`, `System.Collections.Generic`, 0 NuGet) ; les figures matplotlib du twin Python sont rendues en **SVG vectoriel**.

**Navigation** : [<< 04-Computational-Aggregation](ChoixSocial-04-Computational-Aggregation-SAT-Z3-CSharp.ipynb) | [Index](README.md) | [voisinage geometrique : 03f-Deux-Especes-de-Fleches](../GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb)

## Le temoin d'exploitation

Une regle de vote est **manipulable** s'il existe un profil de preferences et un electeur qui peut soumettre un bulletin **insincere** obtenant un resultat qu'il prefere au resultat sincere. Gibbard (1973) et Satterthwaite (1975) ont prouve que **toute regle non dictatoriale** sur au moins 3 alternatives admet une telle manipulation.

La consequence epistemologique est importante : le bulletin insincere est un **temoin** d'exploitation. Quand un electeur ment, il exhibe un bulletin et un vainqueur qui n'aurait pas existe avec un bulletin sincere - et la preuve d'exploitation est constructive. C'est le noyau mathematique propre de ce que Arrow designait comme l'impossibilite d'eviter la strategie sans tomber dans la dictature.

Trois parties, chacune composee d'un exemple guide resolu et d'un exercice a completer :
1. **Une election concrete** - 3 votants, 3 candidats, regle Pluralite. Exemple guide : le vainqueur sincere et sa sensibilite au tiebreak. Exercice : construire un profil a 4 votants ou le tiebreak decide.
2. **Le temoin** - identifier le bulletin insincere qui ameliore le sort d'un electeur. Exemple guide : exhiber le temoin (bulletin + vainqueur + gain). Exercice : generaliser la recherche a un nombre quelconque de votants.
3. **Comparer les regles** - Pluralite vs Borda : meme profil, meme temoin ? Exemple guide : mesurer le cout de la recherche de manipulation. Exercice : trouver un profil a 3 votants ou Borda est manipulable.

Le profil canonique est celui de Satterthwaite (1975), section 4 : 3 votants, preferences lineaires strictes, 3 alternatives. La manipulation existe mais elle est **fragile** (depend du tiebreak). C'est cette fragilite qui rend le theoreme contre-intuitif : on pourrait croire que la manipulation est rare. Elle est structurellement toujours presente.

## 1. Le modele : 3 votants, 3 candidats, regles usuelles

On considere un corps electoral de 3 votants qui doivent choisir parmi 3 alternatives `A, B, C`. Chaque votant `i` a des preferences lineaires strictes (un ordre total sans ex-aequo) sur les alternatives.

**Regles de vote** :
- **Pluralite** : chaque electeur vote pour son candidat prefere. Le candidat avec le plus de voix gagne. En cas d'egalite, on applique un tiebreak lexicographique (`A > B > C`).
- **Borda** : chaque electeur etablit un classement. Le candidat au rang 1 recoit 2 points, rang 2 recoit 1 point, rang 3 recoit 0. Le candidat avec le plus de points gagne.

**Manipulation** : un electeur `i` peut soumettre un bulletin `b'` different de son vrai classement `b` si `regle(b, b_-i, b')` est strictement prefere a `regle(b, b_-i, b)`. Si un tel bulletin existe, l'electeur a interet a mentir, et la regle est **manipulable** sur ce profil.

**Profil canonique (Satterthwaite 1975, variante pedagogique)** :
- V1 : `A > B > C`
- V2 : `B > A > C`
- V3 : `C > B > A`

### Implementation : regles de vote et helpers

Les regles de vote sont implementees en C# pur (BCL .NET 9 : `System.Linq`, `System.Collections.Generic`, 0 NuGet). Les helpers (`PluralityWinner`, `BordaWinner`, `BordaScores`, `RankOf`, `Permutations`) sont exposes dans la classe statique `SocialChoice05`, afin d'etre reutilises par les cellules suivantes.

In [1]:
// --- Regles de vote et helpers (twin C# de ChoixSocial-05-Gibbard-Satterthwaite-Python) ---
// Port C# : BCL .NET 9 (System.Linq, System.Collections.Generic) - 0 NuGet.
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;

public static class SocialChoice05
{
    public static readonly List<string> Alternatives = new() { "A", "B", "C" };

    // Profil canonique (Satterthwaite 1975, simplifie)
    public static readonly List<List<string>> PROFILE = new()
    {
        new() { "A", "B", "C" },  // V1 : A en tete
        new() { "B", "A", "C" },  // V2 : B en tete
        new() { "C", "B", "A" },  // V3 : C en tete (peaks opposes a V1)
    };

    // Pluralite : chaque electeur vote pour son rang 1. Tiebreak = ordre lexicographique.
    public static string PluralityWinner(List<List<string>> ballots, List<string> tiebreak)
    {
        var scores = Alternatives.ToDictionary(a => a, a => 0);
        foreach (var b in ballots) scores[b[0]] += 1;
        // Tiebreak : score d'abord, puis ordre lexicographique
        return scores.OrderByDescending(kv => kv.Value)
                     .ThenBy(kv => tiebreak.IndexOf(kv.Key))
                     .First().Key;
    }

    // Scores Borda : 2 points au rang 1, 1 au rang 2, 0 au rang 3.
    public static Dictionary<string, int> BordaScores(List<List<string>> ballots)
    {
        var scores = Alternatives.ToDictionary(a => a, a => 0);
        foreach (var b in ballots)
            for (int r = 0; r < b.Count; r++)
                scores[b[r]] += 2 - r;
        return scores;
    }

    // Borda : le candidat au plus de points gagne ; departage alphabetique.
    public static string BordaWinner(List<List<string>> ballots)
        => BordaScores(ballots).OrderByDescending(kv => kv.Value)
                                .ThenBy(kv => kv.Key).First().Key;

    // Rang (1-based) d'une alternative dans une preference.
    public static int RankOf(string alternative, List<string> preference)
        => preference.IndexOf(alternative) + 1;

    // Les 6 bulletins possibles (enumeration fixe, equivalente a itertools.permutations).
    public static List<List<string>> Permutations(List<string> items)
    {
        var result = new List<List<string>>();
        void Rec(List<string> prefix, List<string> rest)
        {
            if (rest.Count == 0) { result.Add(prefix); return; }
            for (int i = 0; i < rest.Count; i++)
            {
                var next = new List<string>(rest);
                string pick = next[i];
                next.RemoveAt(i);
                Rec(prefix.Concat(new[] { pick }).ToList(), next);
            }
        }
        Rec(new List<string>(), items);
        return result;
    }

    // Formatage d'une preference.
    public static string Fmt(List<string> pref) => "[" + string.Join(", ", pref) + "]";
}

// Sanity check
var sincereProfile = SocialChoice05.PROFILE.Select(p => p.ToList()).ToList();
Console.WriteLine($"Profil : V1={SocialChoice05.Fmt(sincereProfile[0])}, V2={SocialChoice05.Fmt(sincereProfile[1])}, V3={SocialChoice05.Fmt(sincereProfile[2])}");
Console.WriteLine($"Vainqueur Pluralite sincere : {SocialChoice05.PluralityWinner(sincereProfile, SocialChoice05.Alternatives)}");
Console.WriteLine($"Vainqueur Borda sincere     : {SocialChoice05.BordaWinner(sincereProfile)}");
// Conclusion derivee (PAS de chaine en dur)
var _plurality_w = SocialChoice05.PluralityWinner(sincereProfile, SocialChoice05.Alternatives);
var _borda_w = SocialChoice05.BordaWinner(sincereProfile);
if (_plurality_w == _borda_w)
    Console.WriteLine($"  -> Pluralite et Borda donnent toutes deux {_plurality_w} (coincidence).");
else
    Console.WriteLine($"  -> Pluralite {_plurality_w}, Borda {_borda_w} : divergent.");
Console.WriteLine($"     Rang {_plurality_w} pour V3 : {SocialChoice05.RankOf(_plurality_w, SocialChoice05.PROFILE[2])}.");
Console.WriteLine($"     Rang de A pour V3 : {SocialChoice05.RankOf("A", SocialChoice05.PROFILE[2])} (dernier)");

Profil : V1=[A, B, C], V2=[B, A, C], V3=[C, B, A]


Vainqueur Pluralite sincere : A


Vainqueur Borda sincere     : B


  -> Pluralite A, Borda B : divergent.


     Rang A pour V3 : 3.


     Rang de A pour V3 : 3 (dernier)


## 2. Exemple guide 1 : L'election concrete sous Pluralite

Sous Pluralite sincere :
- V1 vote A (rang 1)
- V2 vote B (rang 1)
- V3 vote C (rang 1)

Le code ci-dessous calcule le vainqueur sous trois ordres de departage, le vainqueur Borda, puis le rang de A pour chaque votant.

### Question 1 : verifier le resultat sincere et sa fragilite

In [2]:
// --- Exemple guide 1 : resultat sincere et sensibilite au tiebreak ---

var sincereBallots = SocialChoice05.PROFILE.Select(p => p.ToList()).ToList();
var tiebreaks = new List<List<string>>
{
    new() { "A", "B", "C" },
    new() { "B", "A", "C" },
    new() { "C", "B", "A" },
};

// Pluralite sincere sous chaque tiebreak
Console.WriteLine("Pluralite sincere sous 3 tiebreaks :");
foreach (var tb in tiebreaks)
    Console.WriteLine($"  Tiebreak {tb[0]}>{tb[1]}>{tb[2]} : vainqueur = {SocialChoice05.PluralityWinner(sincereBallots, tb)}");

// Borda sincere
Console.WriteLine($"\nBorda sincere : vainqueur = {SocialChoice05.BordaWinner(sincereBallots)} (deterministe, pas de tiebreak)");

// Rang du vainqueur pour chaque votant
Console.WriteLine("\nRang du vainqueur A (tiebreak A) pour chaque votant :");
for (int i = 0; i < SocialChoice05.PROFILE.Count; i++)
{
    var vPref = SocialChoice05.PROFILE[i];
    Console.WriteLine($"  V{i + 1} (pref {string.Join(" > ", vPref)}) : rang de A = {SocialChoice05.RankOf("A", vPref)}");
}

// Visualisation : grille des rangs par votant (SVG vectoriel, remplace imshow du twin Python)
var rankingMatrix = SocialChoice05.PROFILE
    .Select(v => SocialChoice05.Alternatives.Select(c => SocialChoice05.RankOf(c, v)).ToList())
    .ToList();
string[] rankColors = { "#2e7d32", "#f9a825", "#c62828" };  // rang 1 = vert (prefere), 2 = orange, 3 = rouge (dernier)
var svgBuilder = new System.Text.StringBuilder();
double cellSize = 60, x0 = 90, y0 = 70;
svgBuilder.AppendLine("<svg xmlns='http://www.w3.org/2000/svg' width='560' height='330' font-family='sans-serif'>");
svgBuilder.AppendLine("<rect width='560' height='330' fill='white'/>");
svgBuilder.AppendLine("<text x='280' y='24' text-anchor='middle' font-size='15'>Rang de chaque candidat dans chaque preference</text>");
svgBuilder.AppendLine("<text x='280' y='44' text-anchor='middle' font-size='12'>(1 = prefere, 3 = dernier)</text>");
for (int j = 0; j < SocialChoice05.Alternatives.Count; j++)
    svgBuilder.AppendLine($"<text x='{x0 + j * cellSize + cellSize / 2}' y='{y0 - 10}' text-anchor='middle' font-size='14'>{SocialChoice05.Alternatives[j]}</text>");
for (int i = 0; i < rankingMatrix.Count; i++)
{
    svgBuilder.AppendLine($"<text x='{x0 - 14}' y='{y0 + i * cellSize + cellSize / 2 + 5}' text-anchor='end' font-size='14'>V{i + 1}</text>");
    for (int j = 0; j < 3; j++)
    {
        int r = rankingMatrix[i][j];
        svgBuilder.AppendLine($"<rect x='{x0 + j * cellSize}' y='{y0 + i * cellSize}' width='{cellSize}' height='{cellSize}' fill='{rankColors[r - 1]}' stroke='black' stroke-width='1'/>");
        svgBuilder.AppendLine($"<text x='{x0 + j * cellSize + cellSize / 2}' y='{y0 + i * cellSize + cellSize / 2 + 6}' text-anchor='middle' font-size='18' font-weight='bold' fill='white'>{r}</text>");
    }
}
for (int r = 0; r < 3; r++)
{
    double ly = y0 + 3 * cellSize + 16 + r * 22;
    string label = r == 0 ? "prefere" : (r == 1 ? "intermediaire" : "dernier");
    svgBuilder.AppendLine($"<rect x='{x0}' y='{ly}' width='18' height='18' fill='{rankColors[r]}' stroke='black'/>");
    svgBuilder.AppendLine($"<text x='{x0 + 26}' y='{ly + 14}' font-size='12'>rang {r + 1} ({label})</text>");
}
svgBuilder.AppendLine("</svg>");
string svgPathSincere = "gt22_sincere.svg";
System.IO.File.WriteAllText(svgPathSincere, svgBuilder.ToString());

Console.WriteLine("\nFigure (SVG vectoriel) : grille des rangs par votant");
Console.WriteLine("        A    B    C");
for (int i = 0; i < rankingMatrix.Count; i++)
    Console.WriteLine($"  V{i + 1}    {rankingMatrix[i][0]}    {rankingMatrix[i][1]}    {rankingMatrix[i][2]}");
Console.WriteLine($"Figure SVG ecrite : {svgPathSincere}");

Console.WriteLine("\nObservation : le vainqueur A est le pire choix pour V3 (rang 3).");
Console.WriteLine("C'est cette insatisfaction qui cree l'opportunite de manipulation.");

Pluralite sincere sous 3 tiebreaks :


  Tiebreak A>B>C : vainqueur = A


  Tiebreak B>A>C : vainqueur = B


  Tiebreak C>B>A : vainqueur = C



Borda sincere : vainqueur = B (deterministe, pas de tiebreak)



Rang du vainqueur A (tiebreak A) pour chaque votant :


... (7 outputs left out)

  V3    3    2    1


Figure SVG ecrite : gt22_sincere.svg



Observation : le vainqueur A est le pire choix pour V3 (rang 3).


C'est cette insatisfaction qui cree l'opportunite de manipulation.


Depouillement : `A=1, B=1, C=1`. **Egalite triple**. Le tiebreak lexicographique designe A, comme le montre la sortie ci-dessus. Mais ce tiebreak est *arbitraire* : avec `B > A > C` le vainqueur est B (meilleur pour V2 et V3), avec `C > B > A` il est C (meilleur pour V3).

L'election sincere **depend du tiebreak**. C'est deja un defaut - une bonne regle devrait etre robuste aux details operationnels - mais ce n'est pas le seul. L'exemple guide 2 exhibe un temoin de manipulation sous le tiebreak `A > B > C` : un electeur peut, par un bulletin insincere, **changer le vainqueur vers un candidat qu'il prefere**.

### Exercice 1 : un profil a 4 votants ou le tiebreak decide

**Contexte.** L'exemple guide 1 montre que le vainqueur sincere de la Pluralite depend du tiebreak sur le profil a 3 votants. Reste a savoir si ce phenomene demande un profil particulier, et combien de votants il faut.

**Objectif.** Construire un profil de 4 votants (4 listes de preferences lineaires strictes sur `A`, `B`, `C`) tel que le vainqueur de la Pluralite change selon l'ordre de departage.

**Indices.**
- Le vainqueur de la Pluralite est le candidat le plus souvent classe premier. Le departage n'intervient que s'il y a une egalite au sommet.
- Deux candidats a egalite au premier rang suffisent pour que le tiebreak choisisse entre eux.
- En C#, une preference est une `List<string>` (du meilleur au pire) : `new() { "A", "B", "C" }`.

**Critere de verification.** La cellule suivante affiche l'ensemble des vainqueurs obtenus sur les six ordres de departage. Le profil est accepte si cet ensemble contient au moins deux candidats.

In [3]:
// --- Exercice 1 : un profil a 4 votants ou le tiebreak decide ---
// TODO etudiant : remplacer null par une liste de 4 preferences lineaires sur A, B, C
// (chaque preference : new() { "A", "B", "C" } - du meilleur au pire)
List<List<string>> PROFIL_TIEBREAK = null;  // TODO etudiant

if (PROFIL_TIEBREAK == null)
{
    Console.WriteLine("Exercice a completer");
}
else
{
    var gagnants = SocialChoice05.Permutations(new List<string> { "A", "B", "C" })
        .Select(tb => SocialChoice05.PluralityWinner(PROFIL_TIEBREAK, tb))
        .Distinct()
        .OrderBy(x => x, StringComparer.Ordinal)
        .ToList();
    Console.WriteLine("Vainqueurs possibles selon le tiebreak : [" + string.Join(", ", gagnants.Select(g => "'" + g + "'")) + "]");
    Console.WriteLine(gagnants.Count > 1 ? "Le tiebreak decide du vainqueur" : "Le vainqueur ne depend pas du tiebreak");
}

Exercice a completer


## 3. Exemple guide 2 : Le temoin - exhiber un bulletin insincere gagnant

V3 prefere `C > B > A`. V1 vote A et V2 vote B : le bulletin de V3 est donc le seul levier. V3 cherche un bulletin `b'` qui fasse gagner un candidat qu'il prefere au vainqueur sincere.

Le code ci-dessous enumere les six bulletins possibles de V3, calcule le vainqueur de chacun et marque ceux qui ameliorent son rang.

### Question 2 : temoin de manipulation Pluralite

In [4]:
// --- Exemple guide 2 : trouver et exhiber le temoin de manipulation ---

// Strategie : pour V3, enumerer tous les bulletins possibles et chercher
// ceux qui produisent un resultat strictement meilleur que A.

int voterGs2 = 2;  // indice 0-based pour V3
var v_prefGs2 = SocialChoice05.PROFILE[voterGs2];
var sincere_winnerGs2 = SocialChoice05.PluralityWinner(
    SocialChoice05.PROFILE.Select(p => p.ToList()).ToList(),
    SocialChoice05.Alternatives);
int sincere_rankGs2 = SocialChoice05.RankOf(sincere_winnerGs2, v_prefGs2);
Console.WriteLine($"V{voterGs2 + 1} preference sincere : {string.Join(" > ", v_prefGs2)}");
Console.WriteLine($"Vainqueur sincere          : {sincere_winnerGs2}");
Console.WriteLine($"Rang de {sincere_winnerGs2} pour V{voterGs2 + 1} : {sincere_rankGs2}");
Console.WriteLine();

// Enumeration : tous les bulletins possibles de V3 (6 permutations)
Console.WriteLine("Recherche de bulletins insincres ameliorant le sort de V3 :");
Console.WriteLine();
Console.WriteLine("Bulletin V3".PadRight(25) + "Vainqueur".PadRight(12) + "Rang pour V3".PadRight(12) + "Gain".PadRight(10));
Console.WriteLine(new string('-', 60));

bool found_manipulation = false;
List<string> manipulation_ballot = null;
string manipulation_winner = null;
int manipulation_rank = 0;
foreach (var insincere in SocialChoice05.Permutations(new List<string> { "A", "B", "C" }))
{
    var ballots = SocialChoice05.PROFILE.Select(p => p.ToList()).ToList();
    ballots[voterGs2] = insincere.ToList();
    string w = SocialChoice05.PluralityWinner(ballots, SocialChoice05.Alternatives);
    int new_rank = SocialChoice05.RankOf(w, v_prefGs2);
    int gain = sincere_rankGs2 - new_rank;  // > 0 si mieux
    string marker = gain > 0 ? "  <-- TEMOIN" : "";
    Console.WriteLine("  " + string.Join(" > ", insincere).PadRight(23)
        + w.PadRight(12) + new_rank.ToString().PadRight(12) + gain.ToString().PadRight(10) + marker);
    if (gain > 0 && !found_manipulation)
    {
        found_manipulation = true;
        manipulation_ballot = insincere.ToList();
        manipulation_winner = w;
        manipulation_rank = new_rank;
    }
}

Console.WriteLine();
if (found_manipulation)
{
    Console.WriteLine("TEMOIN DE MANIPULATION (Pluralite) :");
    Console.WriteLine($"  V{voterGs2 + 1} sincere     : {string.Join(" > ", v_prefGs2)}");
    Console.WriteLine($"  V{voterGs2 + 1} manipule   : {string.Join(" > ", manipulation_ballot)}");
    Console.WriteLine($"  Vainqueur sincere     : {sincere_winnerGs2} (rang {sincere_rankGs2} pour V{voterGs2 + 1})");
    Console.WriteLine($"  Vainqueur manipule    : {manipulation_winner} (rang {manipulation_rank} pour V{voterGs2 + 1})");
    Console.WriteLine($"  GAIN : rang {sincere_rankGs2} -> rang {manipulation_rank}");
}
else
{
    Console.WriteLine("Aucune manipulation trouvee (erreur de profil).");
}

V3 preference sincere : C > B > A


Vainqueur sincere          : A


Rang de A pour V3 : 3


Recherche de bulletins insincres ameliorant le sort de V3 :


Bulletin V3              Vainqueur   Rang pour V3Gain      


------------------------------------------------------------


... (8 outputs left out)

  V3 manipule   : B > A > C


  Vainqueur sincere     : A (rang 3 pour V3)


  Vainqueur manipule    : B (rang 2 pour V3)


  GAIN : rang 3 -> rang 2


V3 vote `B > A > C` au lieu de `C > B > A`. Depouillement : A=1 (V1), B=2 (V2 et V3), C=0. **B gagne**. Le rang de B pour V3 est 2, alors que le vainqueur sincere A y est classe 3e. **Gain : rang 3 -> rang 2**. Le bulletin `B > C > A` produit le meme vainqueur et le meme gain, comme le montre la sortie.

C'est le temoin : un bulletin *concret* (`B > A > C` soumis par V3) + un *vainqueur* (B) + un *gain* (V3 passe de A a B dans son classement).

### Exercice 2 : le temoin a 4 votants

**Contexte.** L'exemple guide 2 cherche un bulletin insincere pour un seul votant (V3) sur un profil fixe. La definition du temoin ne depend ni du votant considere ni du nombre de votants : elle se generalise.

**Objectif.** Ecrire `TemoinPlurality(profil, tiebreak)` qui parcourt tous les votants et tous leurs bulletins possibles. Elle renvoie le premier temoin trouve sous la forme `(indice du votant, bulletin insincere, vainqueur)`, ou `null` s'il n'en existe aucun.

**Indices.**
- Une boucle externe sur les votants, une boucle interne sur `SocialChoice05.Permutations(new List<string> { "A", "B", "C" })`, comme dans l'exemple guide 2.
- Le vainqueur sincere se calcule une seule fois avec `SocialChoice05.PluralityWinner(profil, tiebreak)`.
- Le rang de ce vainqueur dans la preference du votant se lit avec `SocialChoice05.RankOf`.
- Le type de retour attendu est `public static (int votant, List<string> bulletin, string gagnant)?` : `null` quand aucun temoin n'existe.

**Critere de verification.** Sur le profil de l'exemple guide 2, la fonction doit retrouver le temoin de cet exemple (V3, bulletin `B > A > C`). Sur `PROFIL_4`, elle doit renvoyer un temoin, que la cellule affiche.

In [5]:
// --- Exercice 2 : le temoin a 4 votants ---

// TODO etudiant : implementer la recherche du temoin de manipulation.
// Remplacez le corps de TemoinPlurality par :
//   - une boucle externe sur les votants de `profil`,
//   - une boucle interne sur les 6 bulletins possibles (SocialChoice05.Permutations),
//   - le calcul du vainqueur sincere (SocialChoice05.PluralityWinner) et du rang
//     de ce vainqueur pour le votant (SocialChoice05.RankOf),
//   - le retour du premier temoin trouve, ou `null` s'il n'y en a aucun.
public static class Exercice02
{
    public static (int votant, List<string> bulletin, string gagnant)? TemoinPlurality(
        List<List<string>> profil, List<string> tiebreak)
    {
        Console.WriteLine("TODO etudiant : completer TemoinPlurality (parcourir les votants puis leurs 6 bulletins possibles)");
        return null;  // TODO etudiant
    }
}

var PROFIL_4 = new List<List<string>>
{
    new() { "A", "B", "C" },  // V1
    new() { "B", "A", "C" },  // V2
    new() { "B", "A", "C" },  // V3
    new() { "C", "A", "B" },  // V4
};

var temoin = Exercice02.TemoinPlurality(PROFIL_4, SocialChoice05.Alternatives);
if (temoin == null)
{
    Console.WriteLine("Exercice a completer");
}
else
{
    var controle = Exercice02.TemoinPlurality(SocialChoice05.PROFILE.Select(p => p.ToList()).ToList(), SocialChoice05.Alternatives);
    Console.WriteLine("Controle sur le profil de l'exemple guide 2 : "
        + (controle == null ? "aucun temoin"
            : $"V{controle.Value.votant + 1} vote {string.Join(" > ", controle.Value.bulletin)} -> vainqueur {controle.Value.gagnant}"));
    Console.WriteLine($"Temoin : V{temoin.Value.votant + 1} vote {string.Join(" > ", temoin.Value.bulletin)} -> vainqueur {temoin.Value.gagnant}");
}

TODO etudiant : completer TemoinPlurality (parcourir les votants puis leurs 6 bulletins possibles)


Exercice a completer


## 4. Exemple guide 3 : Comparer les regles - Pluralite vs Borda

La manipulation existe-t-elle aussi sous **Borda** sur le meme profil ? Si oui, est-elle plus ou moins difficile a trouver ?

**Hypothese** : Borda prend en compte l'ordre complet, pas seulement le rang 1. Un electeur qui manipule en Borda doit *anticiper* comment son bulletin modifie le score relatif de plusieurs candidats. La manipulation est possible mais demande plus de calcul.

L'exemple **mesure** l'effort de recherche : combien de tentatives un electeur doit-il faire avant de trouver une manipulation ?

### Question 3 : la meme manipulation existe-t-elle sous Borda ?

In [6]:
// --- Exemple guide 3 : Borda est-elle egalement manipulable sur ce profil ? ---

int voterGs3 = 2;
var v_prefGs3 = SocialChoice05.PROFILE[voterGs3];
var sincere_borda_winner = SocialChoice05.BordaWinner(SocialChoice05.PROFILE.Select(p => p.ToList()).ToList());
int sincere_borda_rank = SocialChoice05.RankOf(sincere_borda_winner, v_prefGs3);
Console.WriteLine($"Vainqueur Borda sincere : {sincere_borda_winner} (rang {sincere_borda_rank} pour V{voterGs3 + 1})");
Console.WriteLine();

// Enumeration exhaustive des bulletins de V3
Console.WriteLine("Bulletin V3".PadRight(25) + "Vainqueur Borda".PadRight(18) + "Rang V3".PadRight(10) + "Gain".PadRight(8));
Console.WriteLine(new string('-', 65));
var borda_manipulations = new List<(List<string> bulletin, string winner, int rank)>();
foreach (var insincere in SocialChoice05.Permutations(new List<string> { "A", "B", "C" }))
{
    var ballots = SocialChoice05.PROFILE.Select(p => p.ToList()).ToList();
    ballots[voterGs3] = insincere.ToList();
    string w = SocialChoice05.BordaWinner(ballots);
    int new_rank = SocialChoice05.RankOf(w, v_prefGs3);
    int gain = sincere_borda_rank - new_rank;
    string marker = gain > 0 ? "  <-- TEMOIN" : "";
    Console.WriteLine("  " + string.Join(" > ", insincere).PadRight(23)
        + w.PadRight(18) + new_rank.ToString().PadRight(10) + gain.ToString().PadRight(8) + marker);
    if (gain > 0)
        borda_manipulations.Add((insincere.ToList(), w, new_rank));
}

Console.WriteLine();
if (borda_manipulations.Count > 0)
{
    Console.WriteLine($"Borda : {borda_manipulations.Count} temoin(s) de manipulation trouve(s).");
    foreach (var (ins, w, r) in borda_manipulations)
        Console.WriteLine($"  V3 vote {string.Join(" > ", ins)} -> {w} (rang {r} pour V3)");
}
else
{
    Console.WriteLine("Borda : AUCUN temoin de manipulation pour V3 sur ce profil.");
    Console.WriteLine("Les scores Borda sinceres sont :");
    var scoresGs3 = SocialChoice05.BordaScores(SocialChoice05.PROFILE.Select(p => p.ToList()).ToList());
    foreach (var kv in scoresGs3.OrderByDescending(kv => kv.Value).ThenBy(kv => kv.Key))
        Console.WriteLine($"  {kv.Key} = {kv.Value} points");
    // Conclusion derivee (PAS de chaine en dur)
    var _sortedGs3 = scoresGs3.OrderByDescending(kv => kv.Value).ThenBy(kv => kv.Key).ToList();
    var _top1 = _sortedGs3[0];
    var _top2 = _sortedGs3[1];
    int _marginGs3 = _top1.Value - _top2.Value;
    string _ptsGs3 = _marginGs3 == 1 ? "point" : "points";
    Console.WriteLine($"  -> {_top1.Key} devance {_top2.Key} de {_marginGs3} {_ptsGs3} ({_top1.Value} vs {_top2.Value}).");
}

// Pluralite : meme enumeration, pour la comparaison synthetique (recalcul local)
bool plurality_manipulable = false;
var sincere_plur_winner = SocialChoice05.PluralityWinner(SocialChoice05.PROFILE.Select(p => p.ToList()).ToList(), SocialChoice05.Alternatives);
int sincere_plur_rank = SocialChoice05.RankOf(sincere_plur_winner, v_prefGs3);
foreach (var insincere in SocialChoice05.Permutations(new List<string> { "A", "B", "C" }))
{
    var ballots = SocialChoice05.PROFILE.Select(p => p.ToList()).ToList();
    ballots[voterGs3] = insincere.ToList();
    string w = SocialChoice05.PluralityWinner(ballots, SocialChoice05.Alternatives);
    if (sincere_plur_rank - SocialChoice05.RankOf(w, v_prefGs3) > 0) { plurality_manipulable = true; break; }
}

Console.WriteLine();
Console.WriteLine("=== Comparaison synthetique ===");
Console.WriteLine($"Pluralite : {(plurality_manipulable ? "MANIPULABLE" : "NON MANIPULABLE")} (exploitation du tiebreak)");
Console.WriteLine($"Borda     : {(borda_manipulations.Count > 0 ? "MANIPULABLE" : "NON MANIPULABLE")} (sur ce profil precis)");
Console.WriteLine();
Console.WriteLine("Conclusion nuancee : Pluralite est PLUS facile a manipuler ici, mais Borda");
Console.WriteLine("est structurellement manipulable aussi (Gibbard-Satterthwaite) - le profil");
Console.WriteLine("didactique choisi ici protege Borda par les marges importantes des scores.");
Console.WriteLine("Sur un profil ou les marges sont plus serrees, Borda admet aussi des temoins.");

// Visualisation comparative (SVG vectoriel, remplace les deux subplots matplotlib du twin Python)
var svg2 = new System.Text.StringBuilder();
svg2.AppendLine("<svg xmlns='http://www.w3.org/2000/svg' width='1120' height='430' font-family='sans-serif'>");
svg2.AppendLine("<rect width='1120' height='430' fill='white'/>");
int p1x = 70, p1y = 70, p1w = 440, p1h = 240;
svg2.AppendLine($"<text x='{p1x + p1w / 2}' y='28' text-anchor='middle' font-size='14'>Pluralite : 1 voix suffit pour basculer</text>");
svg2.AppendLine($"<text x='{p1x + p1w / 2}' y='48' text-anchor='middle' font-size='14'>le vainqueur A -&gt; B</text>");
svg2.AppendLine($"<line x1='{p1x}' y1='{p1y + p1h}' x2='{p1x + p1w}' y2='{p1y + p1h}' stroke='black'/>");
svg2.AppendLine($"<line x1='{p1x}' y1='{p1y}' x2='{p1x}' y2='{p1y + p1h}' stroke='black'/>");
svg2.AppendLine($"<text x='18' y='{p1y + p1h / 2}' font-size='12' transform='rotate(-90 18 {p1y + p1h / 2})' text-anchor='middle'>Voix Pluralite</text>");
int[] sincerePlurality = { 1, 1, 1 };
int[] manipPlurality = { 1, 2, 0 };
double bw = 46, gap = 26;
for (int i = 0; i < 3; i++)
{
    double cx = p1x + 45 + i * (2 * bw + gap);
    double h1 = sincerePlurality[i] * 60, h2 = manipPlurality[i] * 60;
    svg2.AppendLine($"<rect x='{cx}' y='{p1y + p1h - h1}' width='{bw}' height='{h1}' fill='lightblue' stroke='blue'/>");
    svg2.AppendLine($"<rect x='{cx + bw + 6}' y='{p1y + p1h - h2}' width='{bw}' height='{h2}' fill='lightcoral' stroke='red'/>");
    svg2.AppendLine($"<text x='{cx + bw / 2}' y='{p1y + p1h - h1 - 5}' text-anchor='middle' font-size='12'>{sincerePlurality[i]}</text>");
    svg2.AppendLine($"<text x='{cx + bw + 6 + bw / 2}' y='{p1y + p1h - h2 - 5}' text-anchor='middle' font-size='12'>{manipPlurality[i]}</text>");
    svg2.AppendLine($"<text x='{cx + bw + 3}' y='{p1y + p1h + 20}' text-anchor='middle' font-size='13'>{SocialChoice05.Alternatives[i]}</text>");
}
svg2.AppendLine($"<rect x='{p1x + 45}' y='{p1y + p1h + 34}' width='16' height='16' fill='lightblue' stroke='blue'/>");
svg2.AppendLine($"<text x='{p1x + 67}' y='{p1y + p1h + 47}' font-size='12'>Sincere</text>");
svg2.AppendLine($"<rect x='{p1x + 145}' y='{p1y + p1h + 34}' width='16' height='16' fill='lightcoral' stroke='red'/>");
svg2.AppendLine($"<text x='{p1x + 167}' y='{p1y + p1h + 47}' font-size='12'>Manipule (V3 vote B)</text>");

int p2x = 620, p2y = 70, p2w = 440, p2h = 240;
var scoresPlot = SocialChoice05.BordaScores(SocialChoice05.PROFILE.Select(p => p.ToList()).ToList());
var rankedPlot = scoresPlot.OrderByDescending(kv => kv.Value).ToList();
int marginPlot = rankedPlot[0].Value - rankedPlot[1].Value;
string ptsPlot = marginPlot == 1 ? "point" : "points";
svg2.AppendLine($"<text x='{p2x + p2w / 2}' y='28' text-anchor='middle' font-size='14'>Borda : marge de {marginPlot} {ptsPlot},</text>");
svg2.AppendLine($"<text x='{p2x + p2w / 2}' y='48' text-anchor='middle' font-size='14'>1 bulletin ne renverse pas</text>");
svg2.AppendLine($"<line x1='{p2x}' y1='{p2y + p2h}' x2='{p2x + p2w}' y2='{p2y + p2h}' stroke='black'/>");
svg2.AppendLine($"<line x1='{p2x}' y1='{p2y}' x2='{p2x}' y2='{p2y + p2h}' stroke='black'/>");
svg2.AppendLine($"<text x='{p2x - 52}' y='{p2y + p1h / 2}' font-size='12' transform='rotate(-90 {p2x - 52} {p2y + p1h / 2})' text-anchor='middle'>Points Borda</text>");
for (int i = 0; i < 3; i++)
{
    string alt = SocialChoice05.Alternatives[i];
    int sc = scoresPlot[alt];
    double cx = p2x + 60 + i * 120;
    double h = sc * 40;
    svg2.AppendLine($"<rect x='{cx}' y='{p2y + p2h - h}' width='70' height='{h}' fill='lightgreen' stroke='darkgreen'/>");
    svg2.AppendLine($"<text x='{cx + 35}' y='{p2y + p2h - h - 6}' text-anchor='middle' font-size='12'>{sc}</text>");
    svg2.AppendLine($"<text x='{cx + 35}' y='{p2y + p2h + 20}' text-anchor='middle' font-size='13'>{alt}</text>");
}
double yWin = p2y + p2h - rankedPlot[0].Value * 40;
svg2.AppendLine($"<line x1='{p2x}' y1='{yWin}' x2='{p2x + p2w}' y2='{yWin}' stroke='red' stroke-dasharray='6 4'/>");
svg2.AppendLine($"<text x='{p2x + 8}' y='{yWin - 6}' font-size='12' fill='red'>Score vainqueur {rankedPlot[0].Key} = {rankedPlot[0].Value}</text>");
svg2.AppendLine("</svg>");
string svgPathCompare = "gt22_compare.svg";
System.IO.File.WriteAllText(svgPathCompare, svg2.ToString());
Console.WriteLine();
Console.WriteLine($"Figure (SVG vectoriel) ecrite : {svgPathCompare}");

Vainqueur Borda sincere : B (rang 2 pour V3)


Bulletin V3              Vainqueur Borda   Rang V3   Gain    


-----------------------------------------------------------------


  A > B > C              A                 3         -1      


  A > C > B              A                 3         -1      


  B > A > C              B                 2         0       


... (13 outputs left out)

est structurellement manipulable aussi (Gibbard-Satterthwaite) - le profil


didactique choisi ici protege Borda par les marges importantes des scores.


Sur un profil ou les marges sont plus serrees, Borda admet aussi des temoins.


Figure (SVG vectoriel) ecrite : gt22_compare.svg


Sur ce profil, Borda n'est pas manipulable par V3 : aucun bulletin ne fait gagner un candidat que V3 prefere a B, qui a 4 points contre 3 pour A et 2 pour C (scores imprimes par la cellule ci-dessus). La comparaison donne donc Pluralite manipulable et Borda non manipulable sur ce profil precis. La fragilite du tiebreak rend Pluralite facilement manipulable ici ; Borda, plus robuste en scores, est immunisee sur ce profil precis mais reste structurellement manipulable. L'exercice 3 cherche un profil a 3 votants ou Borda l'est.

### Exercice 3 : un profil a 3 votants ou Borda est manipulable

**Contexte.** L'exemple guide 3 ne trouve aucun temoin de Borda sur son profil. Le theoreme de Gibbard-Satterthwaite dit qu'il existe des profils ou une regle non dictatoriale sur 3 alternatives est manipulable : reste a exhiber un tel profil pour Borda, avec 3 votants.

**Objectif.** Parmi les profils a 3 votants sur `A`, `B`, `C`, trouver un profil, un votant et un bulletin insincere tels que le votant obtienne, avec ce bulletin, un vainqueur Borda strictement meilleur pour lui.

**Indices.**
- `SocialChoice05.Permutations(new List<string> { "A", "B", "C" })` donne les 6 bulletins possibles ; un produit cartésien (`from ... from ... from ... select ...`) enumere les profils a 3 votants.
- `SocialChoice05.RankOf` donne le rang d'un candidat dans une preference, et `SocialChoice05.BordaWinner` le vainqueur d'un ensemble de bulletins.
- Le departage de `SocialChoice05.BordaWinner` est alphabetique. Un profil dont le vainqueur apres manipulation est unique evite de dependre de ce departage.
- `Exercice03.ScoresBorda` est deja implemente (2 points au rang 1, 1 au rang 2, 0 au rang 3) : reusez-le pour le critere de verification.

**Critere de verification.** La cellule suivante affiche le profil trouve et deux booleens. Le profil est accepte si les deux valent `True`.

In [7]:
// --- Exercice 3 : chercher un profil a 3 votants ou Borda est manipulable ---

public static class Exercice03
{
    // Scores Borda : 2 points au rang 1, 1 au rang 2, 0 au rang 3 (implementation reelle, reutilisable).
    public static Dictionary<string, int> ScoresBorda(List<List<string>> bulletins)
    {
        var scores = SocialChoice05.Alternatives.ToDictionary(a => a, a => 0);
        foreach (var b in bulletins)
            for (int r = 0; r < b.Count; r++)
                scores[b[r]] += 2 - r;
        return scores;
    }

    // TODO etudiant : implementer la recherche du profil manipulable sous Borda.
    // Remplacez le corps par :
    //   - l'enumeration des profils a 3 votants (6 bulletins possibles par votant),
    //   - puis, pour chaque profil et chaque votant, l'enumeration de ses 6 bulletins,
    //   - le test : le vainqueur Borda obtenu avec le bulletin insincere est-il mieux classe
    //     pour le votant que le vainqueur Borda sincere ?
    //   - le retour du premier (profil, votant, bulletin) trouve, ou `null` s'il n'y en a aucun.
    public static (List<List<string>> profil, int votant, List<string> bulletin)? ChercheProfilBordaManipulable()
    {
        Console.WriteLine("TODO etudiant : completer ChercheProfilBordaManipulable (profils a 3 votants, votants, 6 bulletins chacun)");
        return null;  // TODO etudiant
    }
}

var resultat = Exercice03.ChercheProfilBordaManipulable();
if (resultat == null)
{
    Console.WriteLine("Exercice a completer");
}
else
{
    var (profil, votant, bulletin) = resultat.Value;
    var apres = profil.Select((b, i) => i == votant ? bulletin : b).ToList();
    var s = Exercice03.ScoresBorda(apres);
    bool ameliore = SocialChoice05.RankOf(SocialChoice05.BordaWinner(apres), profil[votant])
                   < SocialChoice05.RankOf(SocialChoice05.BordaWinner(profil), profil[votant]);
    int maxScore = s.Values.Max();
    bool unique = s.Count(kv => kv.Value == maxScore) == 1;
    Console.WriteLine("Profil : " + string.Join(" | ", profil.Select(p => string.Join(" > ", p))));
    Console.WriteLine($"V{votant + 1} vote {string.Join(" > ", bulletin)} : amelioration = {ameliore}, vainqueur unique apres manipulation = {unique}");
}

TODO etudiant : completer ChercheProfilBordaManipulable (profils a 3 votants, votants, 6 bulletins chacun)


Exercice a completer


## 5. Conclusion : ce que le temoin prouve

**Trois resultats a retenir** :

1. **Le temoin est constructif** - sur le profil canonique de Satterthwaite, un electeur (V3) peut exhiber un bulletin *concret* (`B > A > C` au lieu de `C > B > A`) qui ameliore son sort sous Pluralite (rang 3 -> rang 2). La preuve d'exploitation n'est pas abstraite : c'est un objet qu'on tient en main.

2. **La regle de vote n'est pas une propriete abstraite** - Pluralite se manipule par exploitation du tiebreak (1 voix suffit), Borda se manipule par exploitation des marges (ici protegee par l'ecart A=4 vs B=3). Le choix d'une regle est un choix sur les **chemins de manipulation** qu'on rend possibles.

3. **Gibbard-Satterthwaite est structurel** - sur *tout* profil avec >=3 alternatives, *toute* regle non dictatoriale admet une manipulation. Le profil choisi ici ne fait qu'*exhiber* un temoin pour Pluralite ; le theoreme dit qu'on en trouverait un pour Borda aussi sur un profil approprie. C'est la portee universelle du resultat.

**Limites du modele** : ce profil est minimal (3 electeurs, 3 alternatives). La litterature etend le resultat a N electeurs et M alternatives, avec des bornes sur le nombre de manipulations selon la structure de la regle (e.g. regles a seuil, regles a veto, regles positionnelles). La complexite de la recherche de manipulation est elle-meme un objet d'etude : certaines regles sont *strategyproof* sur des domaines restreints (e.g. preferences unimodales pour la regle du vote majoritaire a 2 tours), mais pas sur le domaine general.

**Lien avec la science politique** : la manipulation par bulletin insincere est observee empiriquement dans les elections reelles. Le resultat theorique explique pourquoi : il serait etonnant que les electeurs *ne* trouvent pas de manipulation, puisque le theoreme garantit qu'il en existe. Les parades institutionnelles (bulletin secret, depouillement public, education civique) reduisent l'incitation sans elever la barriere computationnelle - c'est la these de Riker (1982) sur l'heresthetic.
## References bibliographiques

- Gibbard, A. (1973). Manipulation of Voting Schemes: A General Result. Econometrica 41(4):587-601.
- Satterthwaite, M.A. (1975). Strategy-proofness and Arrow's Conditions: Existence and Correspondence Theorems for Voting Procedures and Social Welfare Functions. Journal of Economic Theory 10(2):187-217.
- Brandt, F., Conitzer, V., Endriss, U., Lang, J., Procaccia, A. D. (2016). *Handbook of Computational Social Choice*. Cambridge University Press. ISBN 978-1-107-06043-2. **Ancrage ch. 6, §6.2 « Gibbard-Satterthwaite and Its Implications » (Conitzer et Walsh)** — la barriere calculatoire a la manipulation comme angle d'attaque des resultats d'impossibilite : ou la complexite de trouver une manipulation profitable protege en pratique la procedure. Source biblio cluster : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\Social Choice, Voting and Welfare\2016 - Brandt Conitzer Endriss Lang - Handbook of Computational Social Choice.pdf`.

## À retenir

| Leçon | Mesure dans ce carnet |
|---|---|
| Un seul profil, trois vainqueurs : c'est le tiebreak qui décide | `Tiebreak A>B>C : vainqueur = A`, `Tiebreak B>A>C : vainqueur = B`, `Tiebreak C>B>A : vainqueur = C` |
| Deux règles usuelles divergent sur ce même profil | `Vainqueur Pluralite sincere : A` contre `Vainqueur Borda sincere     : B` (`-> Pluralite A, Borda B : divergent.`) |
| La fragilité de la règle se lit dans le rang du gagnant | `Rang de A pour V3 : 3 (dernier)` — le vainqueur de la pluralité est le **pire** choix de V3 |
| Le témoin de manipulation est un objet, pas une intuition | `V3 manipule   : B > A > C` puis `Vainqueur manipule    : B (rang 2 pour V3)` et `GAIN : rang 3 -> rang 2` |
| Une règle peut être protégée par ses marges | `B = 4 points`, `A = 3 points`, `C = 2 points`, donc `-> B devance A de 1 point (4 vs 3).` et `Borda : AUCUN temoin de manipulation pour V3 sur ce profil.` |
| L'absence de témoin sur un profil n'est pas une immunité | `Pluralite : MANIPULABLE (exploitation du tiebreak)` et `Borda     : NON MANIPULABLE (sur ce profil precis)` |

## Vérifiez votre compréhension

**Question 1 — Sur le profil de ce carnet, un votant peut-il améliorer son rang sous Borda ?**

La section 4 affiche le tableau des six bulletins possibles pour V3, avec des gains de `-1` ou `0`, puis la ligne `Borda : AUCUN temoin de manipulation pour V3 sur ce profil.`

<details><summary>Réponse</summary>

Non, sur **ce** profil. Les six bulletins possibles de V3 sont tous évalués, et aucun ne fait gagner à V3 un candidat qu'il préfère à B : les gains sont `-1` ou `0`, jamais positif. La raison est donnée par les scores : `B = 4 points`, `A = 3 points`, `C = 2 points`, donc `-> B devance A de 1 point (4 vs 3).` La marge protège Borda ici. Ce que ce résultat ne dit pas : que Borda est strategyproof. Le carnet le dit lui-même — `Pluralite : MANIPULABLE (exploitation du tiebreak)`, `Borda     : NON MANIPULABLE (sur ce profil precis)` — et l'exercice 3 demande justement le profil où Borda cesse d'être protégé.

</details>

**Question 2 — Pourquoi la fragilité attribuée ici à la règle porte-t-elle sur le tiebreak plutôt que sur la règle elle-même ?**

La section 2 affiche trois vainqueurs pour un seul profil : `Tiebreak A>B>C : vainqueur = A`, `Tiebreak B>A>C : vainqueur = B`, `Tiebreak C>B>A : vainqueur = C`, tandis que la même section affiche `Borda sincere : vainqueur = B (deterministe, pas de tiebreak)`.

<details><summary>Réponse</summary>

Parce que le gain ne vient pas de la règle mais de l'**ordre arbitraire des égalités**. Trois ordres de départage différents donnent trois vainqueurs différents sur un profil inchangé : rien dans les préférences ne décide, c'est la convention qui décide. Borda, sur le même profil, n'a pas besoin de cette convention et rend un vainqueur unique, déterministe. C'est ce qui fait de la pluralité la règle la plus facile à manipuler ici — le mécanisme du témoin est d'ailleurs explicite : `V3 manipule   : B > A > C` déplace le vainqueur de `A` vers `B` et fait passer V3 du rang 3 au rang 2.

</details>

## Pour aller plus loin

- [Index de la sous-série SocialChoice](README.md) — le fil de la sous-série et ce que chaque carnet y ajoute.
- [Index de la série GameTheory](../README.md) — la place de l'impossibilité de manipulation parmi les autres structures de la théorie des jeux.

Les carnets voisins sont cités en prose, sans lien. `ChoixSocial-01-Arrow-Impossibility-Theorem-Python` pose le cadre axiomatique qui précède celui-ci : l'impossibilité de manipulation y est l'aboutissement d'une famille de conditions, alors que ce carnet en montre le mécanisme sur un profil unique. `ChoixSocial-06-Mobius-Aggregation-Pouvoir-Manipulation-Python` attaque le même objet par l'agrégation de Möbius et le pouvoir de manipulation, et `ChoixSocial-10-Manipulation-Strategique-MultiWinner-Python` le mesure sur un espace exhaustif de 216 profils, avec les règles à comité. `ChoixSocial-09-Committees-STV-Monroe-ChamberlinCourant-Python` complète le tableau avec les règles de sélection à plusieurs sièges.

La suite du travail est déjà posée dans le carnet : l'exercice 3 demande le profil à trois votants sur lequel Borda admet un témoin, et l'exercice 1 le profil à quatre votants où c'est le tiebreak seul qui décide du vainqueur.

***

**Provenance.** Ce carnet est le jumeau C# (`.net-csharp`) de `GameTheory/SocialChoice/ChoixSocial-05-Gibbard-Satterthwaite-Python.ipynb` : même numéro, même titre, même fil pédagogique, seuls les noms de fichiers et l'implémentation changent (BCL .NET 9 : `System.Linq`, `System.Collections.Generic` ; figures en SVG vectoriel généré par code, 0 NuGet). Les règles de vote (`SocialChoice05.PluralityWinner`, `SocialChoice05.BordaWinner`, `SocialChoice05.BordaScores`, `SocialChoice05.RankOf`, `SocialChoice05.Permutations`) sont des implémentations pédagogiques from-scratch, équivalentes à celles du jumeau Python. Aucun appel réseau, aucune clé d'API, aucun GPU.